In [24]:
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
import pandas_gbq

In [25]:
gcp_service_acc_file_aluno = r"C:\Users\1597779\data_project-template\secrets\meu-projeto-engdados-e9e71c9de509.json"
gcp_id_do_projeto  = "meu-projeto-engdados"  # ID do seu projeto pessoal criado no GCP.
my_gcp_cred = service_account.Credentials.from_service_account_file(gcp_service_acc_file_aluno)
my_gcp_client = bigquery.Client(credentials=my_gcp_cred)

In [26]:
df_silver = my_gcp_client.query(
    f"SELECT * FROM silver.movies_dc_marvel",
).to_dataframe(create_bqstorage_client=False)

In [27]:
dim_film = (
    df_silver[["FILM", "GENRE", "RATING_MPA", "LANGUAGE", "BREAK_EVEN", "MCU", "MALE_FEMALE_LED", "PHASE"]]
    .drop_duplicates()
    .sort_values(by="FILM")
    .reset_index(drop=True)
)

dim_film["ID_FILM"] = dim_film.index + 1

dim_film.head(5)

,FILM,GENRE,RATING_MPA,LANGUAGE,BREAK_EVEN,MCU,MALE_FEMALE_LED,PHASE,ID_FILM
0,Ant-Man,"Superhero, Action, Comedy, SciFi",PG-13,English,Success,True,Male,2.0,1
1,Ant-Man and the Wasp,"Superhero, Action, Adventure, Comedy, SciFi",PG-13,English,Success,True,Co-starring,3.0,2
2,Ant-Man and the Wasp: Quantumania,"Superhero, Action, Adventure, Comedy, SciFi",PG-13,English,Flop,True,Co-starring,5.0,3
3,Aquaman,"Globetrotting Adventure, Sea Adventure, Superh...",PG-13,English,Success,False,Male,NaN,4
4,Aquaman and the Lost Kingdom,"Action Epic, Superhero, Action, Adventure, Fan...",PG-13,English,Flop,False,Male,NaN,5


In [28]:
dim_staff = (
    df_silver[["DIRECTOR", "WRITER", "STAR"]]
    .drop_duplicates()
    .sort_values(by=["DIRECTOR", "WRITER"])
    .reset_index(drop=True)
)

dim_staff["ID_STAFF"] = dim_staff.index + 1

dim_staff.head(2)

,DIRECTOR,WRITER,STAR,ID_STAFF
0,Alan Taylor,Christopher L Yost,"Chris Hemsworth, Natalie Portman, Tom Hiddleston",1
1,Andy Muschietti,Christina Hodson,"Ezra Miller, Michael Keaton, Sasha Calle",2


In [29]:
dim_franchise = (
    df_silver[["FRANCHISE", "CHARACTER_FAMILY"]]
    .drop_duplicates()
    .sort_values(by=["FRANCHISE", "CHARACTER_FAMILY"])
    .reset_index(drop=True)
)

dim_franchise["ID_FRANCHISE"] = dim_franchise.index + 1

dim_franchise.head(2)

,FRANCHISE,CHARACTER_FAMILY,ID_FRANCHISE
0,DC,Aquaman,1
1,DC,Batman,2


In [30]:
dim_production = (
    df_silver[["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"]]
    .drop_duplicates()
    .sort_values(by=["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"])
    .reset_index(drop=True)
)

dim_production["ID_PRODUCTION"] = dim_production.index + 1

dim_production.head(2)

,PRODUCTION_COMPANY,DISTRIBUTOR,COUNTRY_ORIGIN,FILMING_LOCATION,ID_PRODUCTION
0,20th Century Studios,20th Century Studios,United States,"Medfield State Hospital 45 Hospital Road, Med...",1
1,Amen Ra Films,New Line Cinema,United States,"Vancouver, British Columbia, Canada",2


In [31]:
dim_date = df_silver[["US_RELEASE_DATE"]].dropna().drop_duplicates().sort_values(by="US_RELEASE_DATE").reset_index(drop=True)

dim_date["YEAR"] = dim_date["US_RELEASE_DATE"].dt.year
dim_date["MONTH"] = dim_date["US_RELEASE_DATE"].dt.month
dim_date["QUARTER"] = dim_date["US_RELEASE_DATE"].dt.quarter

dim_date["ID_DATE"] = dim_date["US_RELEASE_DATE"].dt.strftime("%Y%m%d").astype(int)

dim_date.head(2)

,US_RELEASE_DATE,YEAR,MONTH,QUARTER,ID_DATE
0,1978-12-15,1978,12,4,19781215
1,1981-06-19,1981,6,2,19810619


In [32]:
fact_movies = df_silver.copy()

In [33]:
fact_movies = fact_movies.merge(
    dim_film,
    on=["FILM", "GENRE", "RATING_MPA", "LANGUAGE", "BREAK_EVEN", "MCU", "MALE_FEMALE_LED", "PHASE"],
    how="left",
)

In [34]:
fact_movies = fact_movies.merge(
    dim_staff,
    on=["DIRECTOR", "WRITER", "STAR"],
    how="left",
)

In [35]:
fact_movies = fact_movies.merge(
    dim_production,
    on=["PRODUCTION_COMPANY", "DISTRIBUTOR", "COUNTRY_ORIGIN", "FILMING_LOCATION"],
    how="left",
)

In [36]:
fact_movies = fact_movies.merge(
    dim_date,
    on=["US_RELEASE_DATE"],
    how="left"
)

In [39]:
fact_movies = fact_movies[
    [
        "ID_FILM", "ID_STAFF", "ID_PRODUCTION",
        "ID_DATE", "BUDGET_x", "INFLATION_ADJUSTED_BUDGET",
        "GROSS_WORLD_WIDE", "INFLATION_ADJUSTED_WORLDWIDE_GROSS",
        "GROSS_US_CANADA", "DOMESTIC", "BOX_OFFICE_GROSS_OTHER_TERRITORIES",
        "GROSS_OPENING_WEEKEND", "GROSS_TO_BUDGET",
        "RATING_IMDB", "ROTTEN_TOMATOES_CRITIC_SCORE",
        "VOTE", "OSCAR", "NOMINATION", "WIN", "MINUTES"
    ]
]

In [40]:
fact_movies.head(3)

,ID_FILM,ID_STAFF,ID_PRODUCTION,ID_DATE,BUDGET_x,INFLATION_ADJUSTED_BUDGET,GROSS_WORLD_WIDE,INFLATION_ADJUSTED_WORLDWIDE_GROSS,GROSS_US_CANADA,DOMESTIC,BOX_OFFICE_GROSS_OTHER_TERRITORIES,GROSS_OPENING_WEEKEND,GROSS_TO_BUDGET,RATING_IMDB,ROTTEN_TOMATOES_CRITIC_SCORE,VOTE,OSCAR,NOMINATION,WIN,MINUTES
0,16,104,107,20200207,84500000.0,99482958.0,NaN,2.417881e+08,NaN,0.41,121200000.0,NaN,2.43,NaN,78,0,0,0,0,109.0
1,33,104,106,20190607,200000000.0,200000000.0,NaN,2.524430e+08,NaN,0.26,186597000.0,NaN,1.26,NaN,22,0,0,0,0,113.0
2,69,104,105,20190702,160000000.0,160000000.0,NaN,1.131928e+09,NaN,0.35,741395911.0,NaN,7.07,NaN,90,0,0,0,0,129.0


In [42]:
minhas_tabelas = {
    "dim_date": dim_date,
    "dim_film": dim_film,
    "dim_franchise": dim_franchise,
    "dim_production": dim_production,
    "dim_staff": dim_staff,
    "fact_movies": fact_movies,  
}

for key, value in minhas_tabelas.items():
    print(key)

    pandas_gbq.to_gbq(
        value,
        project_id=gcp_id_do_projeto,
        destination_table=f"gold.{key}",
        if_exists="replace",
        credentials=my_gcp_cred,
    )

dim_date
dim_film
dim_franchise
dim_production
dim_staff
fact_movies
